<a href="https://colab.research.google.com/github/angieapol33-bot/FUNDAI-Laboratories-APOLINAR/blob/main/Lab2_Search_Algorithms_APOLINAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Maze Solving Using BFS, DFS, and A* Search

## Fundamentals of Artificial Intelligence

**Name:** Angeline Grace Joy Apolinar
**Course:** BSIT
**Section:** 09282-FUNDAI
**Date:** 08/20/2026

**GitHub URL:** https://github.com/angieapol33-bot/FUNDAI-Laboratories-APOLINAR.git

## Description
This laboratory implements BFS, DFS, and A* Search to solve a maze.
The search results are visualized using matplotlib.


In [15]:
from collections import deque
import heapq
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Tuple, Dict, Optional, Set

## Maze Class

The `Maze` class stores the grid, start position, and goal position.
The value `0` represents an open cell.
The value `1` represents a wall.

In [31]:
class Maze:
    def __init__(self, grid, start, goal):
        self.grid = grid
        self.start = start
        self.goal = goal
        self.rows = len(grid)
        self.cols = len(grid[0]) if self.rows > 0 else 0

    def inside(self, pos):
        r, c = pos
        return 0 <= r < self.rows and 0 <= c < self.cols

    def passable(self, pos):
        if not self.inside(pos):
            return False
        r, c = pos
        return self.grid[r][c] == 0

    def neighbors(self, pos):
        r, c = pos
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        result = []

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            next_pos = (nr, nc)

            if self.passable(next_pos):
                result.append(next_pos)

        return result

## SearchResult Class

The `SearchResult` class stores the output of each search algorithm.
It includes the algorithm name, success status, path, visited order, and parent map.

In [30]:
class SearchSolver:
    def __init__(self, maze):
        self.maze = maze

    def _reconstruct_path(self, parent, goal):
        path = []
        current = goal

        while current is not None:
            path.append(current)
            current = parent[current]

        path.reverse()
        return path

    def _heuristic(self, a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

## SearchSolver Class

The `SearchSolver` class contains the implementation of BFS, DFS, and A* Search.

In [41]:
class SearchSolver:
    def __init__(self, maze):
        self.maze = maze

    def _reconstruct_path(self, parent, goal):
        path = []
        current = goal

        while current is not None:
            path.append(current)
            current = parent[current]

        path.reverse()
        return path

    def _heuristic(self, a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def bfs(self):
        start = self.maze.start
        goal = self.maze.goal

        if not self.maze.passable(start) or not self.maze.passable(goal):
            return SearchResult("BFS", False, [], [], {start: None}, start)

        frontier = deque([start])
        visited = {start}
        parent = {start: None}
        visited_order = []
        success = False

        while frontier:
            current = frontier.popleft()
            visited_order.append(current)

            if current == goal:
                success = True
                break

            for neighbor in self.maze.neighbors(current):
                if neighbor not in visited:
                    visited.add(neighbor)
                    parent[neighbor] = current
                    frontier.append(neighbor)

        path = self._reconstruct_path(parent, goal) if success else []

        return SearchResult(
            algorithm="BFS",
            success=success,
            path=path,
            visited_order=visited_order,
            parent=parent,
            root=start
        )


## Sample Maze

The maze uses:

- `0` for open cells
- `1` for walls

In [48]:
grid = [
    [0, 0, 0, 0, 1, 0, 0],
    [1, 1, 0, 1, 1, 0, 1],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0]
]

start = (0, 0)
goal = (6, 6)

maze = Maze(grid, start, goal)
solver = SearchSolver(maze)

In [47]:
bfs_result.summary()
dfs_result.summary()
astar_result.summary()

=== BFS Summary ===
Success          : True
Path length      : 12 steps
Nodes expanded   : 30
Visited cells    : 31
Final path       : [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (3, 6), (4, 6), (5, 6), (6, 6)]

=== DFS Summary ===
Success          : True
Path length      : 16 steps
Nodes expanded   : 17
Visited cells    : 20
Final path       : [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6)]

=== A* Summary ===
Success          : True
Path length      : 12 steps
Nodes expanded   : 14
Visited cells    : 14
Final path       : [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (3, 6), (4, 6), (5, 6), (6, 6)]



In [20]:
def visualize_path(maze: Maze, result: SearchResult, title: str = None):
    """
    Visualize the maze, visited cells, and final path.
    """
    grid = np.array(maze.grid, dtype=float)


    display = np.zeros((maze.rows, maze.cols, 3))

    for r in range(maze.rows):
        for c in range(maze.cols):
            if maze.grid[r][c] == 1:
                display[r, c] = [0.1, 0.1, 0.1]
            else:
                display[r, c] = [1.0, 1.0, 1.0]


    for r, c in result.visited:
        if (r, c) != maze.start and (r, c) != maze.goal:
            display[r, c] = [0.6, 0.8, 1.0]


    for r, c in result.path:
        if (r, c) != maze.start and (r, c) != maze.goal:
            display[r, c] = [1.0, 0.3, 0.3]


    sr, sc = maze.start
    gr, gc = maze.goal
    display[sr, sc] = [0.2, 0.8, 0.2]
    display[gr, gc] = [1.0, 0.85, 0.0]

    plt.figure(figsize=(7, 7))
    plt.imshow(display, origin='upper')
    plt.title(title or f"{result.algorithm_name} Path")
    plt.xticks(range(maze.cols))
    plt.yticks(range(maze.rows))
    plt.grid(True, color='gray', linewidth=0.5)
    plt.show()

## Search Tree Documentation

The search tree is generated from the parent map.
Each child node is connected to its predecessor.



In [37]:
def search_tree_text(result, max_lines=100):
    children = {}

    for child, predecessor in result.parent.items():
        if predecessor is not None:
            if predecessor not in children:
                children[predecessor] = []
            children[predecessor].append(child)

    for node in children:
        children[node].sort()

    lines = []

    def build(node, depth):
        if len(lines) >= max_lines:
            return

        lines.append(" " * depth + str(node))

        for child in children.get(node, []):
            build(child, depth + 1)

    build(result.root, 0)

    if len(lines) >= max_lines:
        lines.append("... search tree truncated for report")

    return "\n".join(lines)

### BFS
BFS explores the maze level by level. It is useful for finding a shortest path when all move costs are equal.

### DFS
DFS explores one branch deeply before backtracking. It may find a solution quickly, but the path may not be short.

### A* Search
A* uses both actual cost from the start and estimated cost to the goal. With Manhattan distance, it is often more efficient than BFS.

### Observation
While both BFS and A* Search successfully located optimal paths of equal minimal length, A* demonstrated superior efficiency by expanding fewer nodes thanks to its heuristic-guided search. Conversely, DFS traversed significantly deeper into sub-paths before backtracking, resulting in an unoptimized, longer path and unnecessary state expansions.

## Guide Questions and Answers

### 1. Which algorithm found the shortest path?
**Answer:** Both **BFS** and **A* Search** successfully identified optimal, shortest paths, whereas DFS produced a significantly longer path due to its depth-first traversal strategy.

### 2. Which algorithm expanded the fewest nodes?
**Answer:** **A* Search** evaluated the lowest total count of states, leveraging its Manhattan heuristic function to prioritize paths leading directly toward the target goal.

### 3. Why is Manhattan distance suitable for this maze?
**Answer:** Movement inside this grid is constrained strictly to orthogonal steps (up, down, left, right) without diagonal traversal, making the sum of absolute coordinate differences an exact, admissible, and consistent distance metric.

### 4. How does the parent map help reconstruct the path?
**Answer:** The parent dictionary maintains a key-value linkage mapping every discovered node back to its predecessor, allowing the program to backtrack step-by-step from the target coordinate back to the origin node.

### 5. Why is repeated-state checking important?
**Answer:** It prevents search algorithms from falling into infinite processing loops, drastically cuts unnecessary state re-evaluations, and reduces the memory footprint required during graph expansion.

## Step 20: Reflection

### Reflection

In this laboratory I implemented three classic uninformed and informed search algorithms to solve a maze.  

I observed that:
- BFS is simple and reliable for shortest paths but can consume a lot of memory on large mazes.
- DFS is memory-efficient and easy to code with a stack, yet the path it returns is often far from optimal.
- A* with the Manhattan distance heuristic strikes a good balance: it still finds the optimal path while expanding significantly fewer nodes than BFS.

The parent-map technique made path reconstruction straightforward and also served as a clear way to document the search tree.  
Visualizing the visited cells and final path with matplotlib helped me understand how each algorithm explores the state space differently.

Overall, this lab strengthened my understanding of the trade-offs between completeness, optimality, time complexity, and space complexity in search algorithms.